# Import libraries

In [ ]:
import os
import json
from tempfile import TemporaryDirectory
import zipfile

from kagglesdk.blobs.types.blob_api_service import ApiBlobType
from kagglehub.gcs_upload import UploadDirectoryInfo, _upload_file
from kagglehub.handle import parse_dataset_handle
from kagglehub.datasets_helpers import create_dataset_or_version
from huggingface_hub import hf_hub_download

# Set up environment variable

In [ ]:
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
os.environ["HF_TOKEN"] = user_secrets.get_secret("HF_TOKEN")

# Config

In [ ]:
ROOT_DIR = "data"
FILE_LIST = "/kaggle/input/datasets/ldthanh/epic-kitchens-100-p2-2/file_list.json"
TARGET_PARTITIONS = ['P12', 'P02', 'P03', 'P10']

HF_DATASET_REPO = "a1raman/epic_kitchens_100"

HANDLE = "ldthanh/epic-kitchens-100-p2-2"
VERSION_NOTES = "Upload data"

# Define utilities

In [ ]:
def create_directory_structure():
    with open(FILE_LIST, "r") as f:
        file_list = json.load(f)

    for p in file_list:
        if p not in TARGET_PARTITIONS:
            continue

        os.makedirs(os.path.join(ROOT_DIR, p, "videos"), exist_ok=True)

        for f in file_list[p]:
            with open(os.path.join(ROOT_DIR, p, "videos", f + ".MP4"), "w") as fp:
                pass  # Create an empty file with the same name to maintain the structure


def upload_data():
    root_dict = UploadDirectoryInfo(name="root")
    for dir_name in sorted(os.listdir(ROOT_DIR)):
        dir_path = os.path.join(ROOT_DIR, dir_name)
        for root, _, files in os.walk(dir_path):
            # Path of the current folder relative to the base folder
            path = os.path.relpath(root, ROOT_DIR)

            # Navigate or create the dictionary path to the current folder
            current_dict = root_dict
            if path != ".":
                for part in path.split(os.sep):
                    # Find or create the subdirectory in the current dictionary
                    for subdir in current_dict.directories:
                        if subdir.name == part:
                            current_dict = subdir
                            break
                    else:
                        # If the directory is not found, create a new one
                        new_dir = UploadDirectoryInfo(name=part)
                        current_dict.directories.append(new_dir)
                        current_dict = new_dir

            # Add file tokens to the current directory in the dictionary
            for file in sorted(files):
                file_path = os.path.join(root, file)
                with TemporaryDirectory() as temp_dir:
                    # Download the actual file from Hugging Face Hub
                    print(f"Downloading {file_path} from Hugging Face Hub...")
                    downloaded_path = hf_hub_download(
                        repo_id=HF_DATASET_REPO,
                        filename=os.path.relpath(file_path, ROOT_DIR),
                        local_dir=temp_dir,
                        repo_type="dataset",
                    )
                    print(f"Downloaded {downloaded_path} for {file_path}")

                    # Upload the file to Kaggle
                    token = _upload_file(
                        os.path.realpath(downloaded_path),
                        item_type=ApiBlobType.DATASET,
                        quiet=False
                    )
                    os.remove(os.path.realpath(downloaded_path))
                    if token:
                        current_dict.files.append(token)
                    else:
                        print(f"WARNING: Failed to upload {dir_name}, skipping.")

    return root_dict


def create_dataset(root_dict: UploadDirectoryInfo):
    dataset_handle = parse_dataset_handle(HANDLE)
    print(f"Uploading Dataset {dataset_handle.to_url()} ...")
    if dataset_handle.is_versioned():
        is_versioned_exception = "The dataset handle should not include the version"
        raise ValueError(is_versioned_exception)

    create_dataset_or_version(dataset_handle, root_dict, VERSION_NOTES)

# Main

In [ ]:
create_directory_structure()
root_dict = upload_data()
create_dataset(root_dict)